<h1 style="text-align: center;">Deep Learning Project</h1>

** **

## <p style="text-align: center;"><i>02 - Pre-Processing </i></p>

** **
<p style="text-align: center;">
David Cascão - 20240851 <br>
Jan-Louis Schneider - 20240506 <br>
Emir Kamiloglu - 20240945 <br>
Marta Boavida - 20240519 <br>
Sofia Gomes - 20240848
</p>


## <span style="color:#FF007F">  Notebook  </span> 

# <span style="color:yellow">  ACABAR  </span> 

In this notebook, 


## <span style="color:#FF007F"> Table of Contents </span>
# <span style="color:yellow">  change everything  </span> 

<a class="anchor" id="top"></a>

1. [Import Libraries](#one-bullet) <br>

2. [Import Datasets](#two-bullet) <br>

   2.1 [Create a Small Sample from the Training Set](#two-one-bullet) <br>

   2.2 [Extracted Feature DataFrame (features_df)](#two-two-bullet) <br>

3. [Visualizations](#three-bullet) <br>

   3.1 [Boxplots of RGB and HSV Features per Class](#three-one-bullet) <br>

   3.2 [RGB Channels](#three-two-bullet) <br>

   3.3 [HSV Features](#three-three-bullet) <br>

   3.4 [Image Quality Metrics: Sharpness](#three-four-bullet) <br>

   3.5 [Visualizing Extremes in Sharpness](#three-five-bullet) <br>

   3.6 [Shape Analysis: Morphological Features](#three-six-bullet) <br>

   3.7 [Shape Extremes](#three-seven-bullet) <br>

   3.8 [Foreground Composition: Animal-to-Background Ratio](#three-eight-bullet) <br>

   3.9 [Foreground Ratio Extremes](#three-nine-bullet) <br>

4. [CLIP-Based Image Filtering](#four-bullet) <br>

   4.1 [Reclassify Non-Animal Samples Using CLIP Predictions](#four-one-bullet) <br>
   
   4.2 [Class Distribution](#four-two-bullet) <br>
   
5. [Augmentation as a Data Balancer](#five-bullet) <br>

   


<a class="anchor" id="one-bullet"></a>

## <span style="color:#FF007F"> 1. Import Libraries</span>

<a href="#top">Top &#129033;</a>

The first step is to install some necessary packages and pretrained models.

In [ ]:
# # Uninstall potential conflicts
# !pip uninstall -y clip torchvision

# # Install CLIP and dependencies
# !pip install ftfy regex tqdm
# !pip install git+https://github.com/openai/CLIP.git

# # Install other requirements
# !pip install rembg[gpu] pillow matplotlib torch torchvision pandas numpy opencv-python

# # For data loading and augmentation
# !pip install albumentations


Found existing installation: clip 1.0
Uninstalling clip-1.0:
  Successfully uninstalled clip-1.0
Found existing installation: torchvision 0.21.0
Uninstalling torchvision-0.21.0:
  Successfully uninstalled torchvision-0.21.0


  Cloning https://github.com/openai/CLIP.git to c:\users\asus.laptop-sfdpa4g4\appdata\local\temp\pip-req-build-t4fa3h_6
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached torchvision-0.21.0-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
Using cached torchvision-0.21.0-cp312-cp312-win_amd64.whl (1.6 MB)
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369594 sha256=47933794bd53fc2208d730b2162f97eb9a7e781d656a2a9362d4bc6db786fb0a
  Stored in directory: C:\Users\ASUS.LAPTOP-SFDPA4G4\AppData\Local\Temp\pip-ephem-wheel-cache-28gkqjt4\wheels\35\3e\df\3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git 'C:\Users\ASUS.LAPTOP-SFDPA4G4\AppData\Local\Temp\pip-req-build-t4fa3h_6'


In [7]:
import os
import clip
import torch
import pandas as pd
import numpy as np
from PIL import Image
import cv2
from torchvision import transforms
from rembg import remove
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import random
import PIL.Image
from collections import defaultdict
PIL.Image.MAX_IMAGE_PIXELS = 115600000 * 2  # Double the current limit

<a class="anchor" id="two-bullet"> 

## <span style="color:#FF007F"> 2. Import Datasets</span> 

<a href="#top">Top &#129033;</a>

Next, we start by importing the dataset.

In [ ]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# imbalanced
train_df_imbalanced = pd.read_csv("../data_preproc/train_df_clip.csv")
val_df_imbalanced = pd.read_csv("../data_preproc/val_df_clip.csv")
test_df_imbalanced = pd.read_csv("../data_preproc/test_df_clip.csv")
#balanced
#train_df = pd.read_csv("../data_preproc/train_df_clip_augmentation.csv")
#val_df = pd.read_csv("../data_preproc/val_df_clip_augmentation.csv")
#test_df = pd.read_csv("../data_preproc/test_df_clip_augmentation.csv")

# Simple Dataset Class
class SimpleDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        self.labels = df['true_class'].astype('category').cat.codes.values
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# Basic transforms
basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Create datasets
train_data = SimpleDataset(train_df_imbalanced, transform=basic_transform)
val_data = SimpleDataset(val_df_imbalanced, transform=basic_transform)
test_data = SimpleDataset(test_df_imbalanced, transform=basic_transform)

# Create loaders
BATCH_SIZE = 32
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

# Test one batch
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")  # Should be [32, 3, 224, 224]
print(f"Labels shape: {labels.shape}")  # Should be [32]

Batch shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])


<a class="anchor" id="two-one-bullet"></a>

### 2.1 Create a Small Sample from the Training Set

<a href="#top">Top &#129033;</a>

To work with a smaller, more manageable subset of the training data (useful for quick experiments or visualization), this snippet randomly selects 1% of the full dataset.

In [27]:
sample_df = train_df.sample(frac=0.01, random_state=42).reset_index(drop=True)
sample_paths = get_image_paths(sample_df)

<a class="anchor" id="two-bullet"></a>

## <span style="color:#FF007F">2. something</span>

<a href="#top">Top &#129033;</a>
